# Disentanglement + strict cross-crop SupCon
ToTnki baseline foundation: two independent frozen timm DINOv2 ViT-B/14 models. Only common/ and installed libraries are required.

Formal controls: common seed, exact common train/validation, ImageNet transforms,20 epochs,batch16,FP32,AdamW crop1e-4/disease3e-4,decay1e-4,smoothing0.1,cosine. No custom holdout, seed grid, sampler, augmentation or scorer search. Old trained checkpoints are not used.


## 1. Setup
Use the same Python3.11 environment and root requirements as the baseline. B additionally needs `transformers==4.52.2` and `huggingface_hub<1` (`python -m pip install transformers==4.52.2 "huggingface_hub<1"`). Both notebooks contain their own code: do not import old experiment folders. Default run controls perform an audit only.


In [ ]:
import copy, gc, hashlib, json, math, random, subprocess, sys, time
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from uuid import uuid4
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as shared
from common.dataset import get_transforms, get_loaders, get_pairs, get_crop_and_disease_names
from common.evaluate import top_k_accuracy, harmonic_mean
MODEL = 'vit_base_patch14_dinov2.lvd142m'

def sha(path):
    with Path(path).open('rb') as stream: return hashlib.file_digest(stream, 'sha256').hexdigest()


def digest_frame(frame):
    return hashlib.sha256(frame[['image','crop','disease','pair_row']].sort_values('image').to_csv(index=False,lineterminator='\n').encode()).hexdigest()
def write(path,value):
    with Path(path).open('x',encoding='utf8') as stream:json.dump(value,stream,indent=2,allow_nan=False)


def fingerprint(value):return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()


def read(path):return json.loads(Path(path).read_text(encoding='utf8'))


def seed_all(seed=None):
    if seed is not None and seed!=shared.SEED:raise ValueError('Use common.SEED only.')
    shared.set_seed(shared.SEED)


def environment():
    result=dict(python=sys.version,cuda_build=torch.version.cuda,
        device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
    for name in ['torch','torchvision','transformers','huggingface_hub','pandas','numpy']:
        try:result[name]=version(name)
        except PackageNotFoundError:result[name]=None
    for name,args in [('git_commit',['git','rev-parse','HEAD']),('git_dirty',['git','status','--porcelain'])]:
        try:result[name]=subprocess.check_output(args,text=True,stderr=subprocess.DEVNULL).strip()
        except (OSError,subprocess.CalledProcessError):result[name]='unavailable'
    result['determinism']='seeded CPU/CUDA, cuDNN deterministic; no guarantee of cross-device/version bit equality'
    return result

NOTEBOOK_NAMES = ['approach3a_dino_disentangle_supcon.ipynb','approach3b_dino_text_alignment.ipynb']
def code_hashes(family):
    if family not in ('A','B'):raise ValueError('Unknown notebook family')
    result={}
    for name in [NOTEBOOK_NAMES[0 if family=='A' else 1]]:
        path=ROOT/'approaches'/name
        notebook=json.loads(path.read_text(encoding='utf8'))
        sources=[''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code' and 'implementation' in c['metadata'].get('tags',[])]
        result['approaches/'+name]=fingerprint(sources)
    for name in ['config.py','dataset.py','evaluate.py']:
        result['common/'+name]=sha(ROOT/'common'/name)
    return result


In [ ]:
# Edit only these run controls. Formal baseline settings are locked below.
MODE = 'smoke'  # 'full' after real-weight smoke succeeds
ALLOW_DOWNLOAD = False  # True explicitly permits public pretrained downloads
RUN_TRAIN = False
RUN_FINAL = False
EXISTING_RUN = None  # Path to a completed frozen run, if reopening this notebook
OTHER_RUN = None     # Completed other-family run; needed for the formal joint lock
COMPARISON_LOCK = None  # Or an already created lock path
CONFIG = json.loads(r'''{
  "protocol": "323-development-333-final",
  "image_backbone": "vit_base_patch14_dinov2.lvd142m",
  "dino_revision": "main",
  "text_revision": "main",
  "epochs": 20,
  "batch_size": 16,
  "accumulation_steps": 1,
  "weight_decay": 0.0001,
  "precision": "float32",
  "partial_blocks": 0,
  "gradient_checkpointing": false,
  "gradient_clip": null,
  "hidden_dim": 256,
  "feature_dim": 128,
  "supcon_temperature": 0.1,
  "text_batch_size": 16,
  "score_grid": [
    [
      0.5,
      0.25,
      0.25
    ]
  ],
  "corruptions": [
    "brightness",
    "blur",
    "noise"
  ],
  "pair_templates": [
    "a photo of a {crop} leaf with {disease}",
    "a field photo of {crop} showing {disease} symptoms",
    "a close-up leaf of {crop} affected by {disease}"
  ],
  "healthy_templates": [
    "a photo of a healthy {crop} leaf",
    "a field photo of a healthy {crop} plant",
    "a close-up healthy {crop} leaf"
  ],
  "label_smoothing": 0.1,
  "crop_lr": 0.0001,
  "disease_lr": 0.0003
}''')


## 2. Data
Direct common.get_loaders:7223 train/777 validation. common provides the seed, transforms and candidate order. Only training CSV metadata is accessed. No extra holdout or batch sampler. num_workers0/pin_memoryFalse are the explicit ToTnki runtime settings; common files remain unchanged.


In [ ]:
def development():
    train_loader,val_loader,_,_=get_loaders(1,norm='imagenet',debug=False)
    train,validation=train_loader.dataset.frame,val_loader.dataset.frame
    pairs=get_pairs()
    if len(train)+len(validation)!=8000 or pairs.shape!=(shared.NUM_SEEN_PAIRS,2):raise ValueError('Unexpected common dataset contract.')
    if (len(train),len(validation)) != (7223,777): raise ValueError('Original split changed.')
    frames = dict(fit=train, validation=validation)
    if not train.image.is_unique or not validation.image.is_unique or set(train.image)&set(validation.image):
        raise ValueError('Duplicate/overlapping development image IDs.')
    names = get_crop_and_disease_names()
    if (len(names[0]),len(names[1])) != (55,175): raise ValueError('Training concept vocabulary changed.')
    return frames, pairs, names, dict(protocol='common_7223_777_v2',train_csv_sha256=sha(shared.TRAIN_CSV),
        signatures={k:digest_frame(v) for k,v in frames.items()}, development_candidates=pairs.tolist(),
        official_metadata_read=False, original_train_images=7223, original_validation_images=777,
        selection='Top1 on all 777 common validation images, over 323 candidates; earliest epoch on ties',
        plant_or_burst_group_ids='not provided in seven-column metadata; exact-byte duplicate checks only')


def duplicate_audit(groups):
    hashes={};owners={};duplicates=[]
    for group,frame in groups.items():
        for name in frame.image:
            digest=sha(shared.IMAGE_DIR/name);hashes[name]=digest
            if digest in owners:
                previous_group,previous_name=owners[digest]
                if previous_group!=group: raise ValueError('Exact image duplicate across partitions: '+previous_name+' / '+name)
                duplicates.append([previous_name,name])
            else:owners[digest]=(group,name)
    return dict(image_sha256=hashes, within_partition_duplicates=duplicates)


def loader(frame,config,training=False,epoch=0,seed=None):
    from common.dataset import _make_loader
    transform=get_transforms('imagenet')[0 if training else 1]
    # Exact ToTnki runtime override; do not edit common or replace its sampler.
    workers=shared.NUM_WORKERS
    try:
        shared.NUM_WORKERS=0
        result=_make_loader(frame,transform,'imagenet',config['batch_size'],shuffle=training,
                            role='train' if training else 'seen_dev')
    finally:shared.NUM_WORKERS=workers
    result.pin_memory=False  # ToTnki baseline setting; common transforms/sampler unchanged.
    return result


def official(dev_pairs):
    # Caller must verify the frozen manifest BEFORE entering this function.
    pairs=get_pairs(final_evaluation=True)
    if pairs.shape!=(333,2) or not torch.equal(pairs[:323],dev_pairs):raise ValueError('Final candidate order changed.')
    from common.dataset import get_final_test_loaders
    seen,unseen=get_final_test_loaders(1,final_evaluation=True)
    return pairs,dict(seen=seen.dataset.frame,unseen=unseen.dataset.frame)


In [ ]:
frames, development_pairs, concept_names, data_manifest = development()
print('common seed:', shared.SEED)
print('train / validation:', len(frames['fit']), len(frames['validation']))
print('development candidates:', len(development_pairs))


## 3. Model
A adds branch projections, decorrelation and strict cross-crop disease SupCon. The baseline variant retains ToTnki's original linear heads; ce uses the new heads without proposed losses.
The resolved public weight file is explicitly bound to both timm loads. Structural changes are evaluated separately from method effects.


In [ ]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path


class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=shared.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            model.requires_grad_(False);model.head.requires_grad_(True)
            return model
        self.crop=pinned_source_model(shared.NUM_CROPS)
        self.disease=pinned_source_model(shared.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            weight_sha256=sha(weights),architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    def policy(self,blocks,checkpointing=True):
        if blocks!=0:raise ValueError('ToTnki baseline freezes both visual towers; keep partial_blocks=0.')
        self.requires_grad_(False)
    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)


class TextOnly:
    """No CLIPModel, CLIPVisionModel, OpenCLIP model, or image encoder is created."""
    def __init__(self,config,device,allow_download=False,revision=None):
        from transformers import CLIPTextModelWithProjection, CLIPTokenizerFast
        path=snapshot('openai/clip-vit-base-patch16',revision or config['text_revision'],allow_download,
            ['*.json','vocab.json','merges.txt','model.safetensors','pytorch_model.bin'])
        self.model=CLIPTextModelWithProjection.from_pretrained(path,local_files_only=True).to(device).eval().requires_grad_(False)
        self.tokenizer=CLIPTokenizerFast.from_pretrained(path,local_files_only=True)
        if any('vision' in name for name,_ in self.model.named_modules()):raise RuntimeError('A CLIP vision encoder was instantiated.')
        self.dimension=self.model.config.projection_dim;self.device=device
        if self.dimension!=512:raise ValueError('Expected the CLIP B/16 512-dimensional text projection.')
        weight=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        self.metadata=dict(repository='openai/clip-vit-base-patch16',revision=path.name,weight_sha256=sha(weight),
            class_name=type(self.model).__name__,dimension=self.dimension,trainable_parameters=0,clip_image_encoder=False)
    @torch.no_grad()
    def encode(self,prompts,size):
        self.assert_frozen();result=[]
        for start in range(0,len(prompts),size):
            tokens=self.tokenizer(prompts[start:start+size],padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
            result.append(F.normalize(self.model(**tokens).text_embeds.float(),dim=1))
        return torch.cat(result)
    def assert_frozen(self):
        if self.model.training or any(p.requires_grad or p.grad is not None for p in self.model.parameters()):
            raise RuntimeError('CLIP text encoder must be eval, frozen and gradient-free.')


def text_banks(text,pairs,names,config,ensemble=False):
    crops,diseases=names
    def clean(x):return x.replace('_',' ')
    pair_groups=[]
    for c,d in pairs.tolist():
        if clean(diseases[d]).strip().casefold()=='healthy':
            templates=config['healthy_templates'] if ensemble else config['healthy_templates'][:1]
        else:templates=config['pair_templates'] if ensemble else config['pair_templates'][:1]
        pair_groups.append([t.format(crop=clean(crops[c]),disease=clean(diseases[d])) for t in templates])
    flat=[p for group in pair_groups for p in group];encoded=text.encode(flat,config['text_batch_size'])
    bank=[];offset=0
    for group in pair_groups:
        bank.append(F.normalize(encoded[offset:offset+len(group)].mean(0),dim=0));offset+=len(group)
    crop_prompts=[f'a photo of a {clean(crops[i])} plant' for i in range(shared.NUM_CROPS)]
    disease_prompts=[('a photo of a healthy leaf' if clean(diseases[i]).casefold()=='healthy' else f'a leaf showing {clean(diseases[i])}') for i in range(shared.NUM_DISEASES)]
    return [torch.stack(bank),text.encode(crop_prompts,config['text_batch_size']),text.encode(disease_prompts,config['text_batch_size'])],dict(pair=pair_groups,crop=crop_prompts,disease=disease_prompts)


def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))


class Model(nn.Module):
    def __init__(self,backbone,family,config,text_dimension=512,banks=None):
        super().__init__();self.backbone=backbone;self.family=family
        width=backbone.width;hidden=config['hidden_dim'];dim=config['feature_dim'] if family=='A' else text_dimension
        self.crop_projection=projection(width,hidden,dim);self.disease_projection=projection(width,hidden,dim)
        self.crop_head=nn.Linear(dim,shared.NUM_CROPS);self.disease_head=nn.Linear(dim,shared.NUM_DISEASES)
        if family=='B':
            self.pair_projection=projection(width,hidden,dim)
            self.log_temperature=nn.Parameter(torch.tensor(math.log(1/.07)))
            for i,bank in enumerate(banks or []):self.register_buffer('text'+str(i),bank.detach())
    def install_text(self,banks):
        for i,bank in enumerate(banks):setattr(self,'text'+str(i),bank.detach())
    def forward(self,pixels,aligned=False):
        h=self.backbone(pixels)
        hc,hd=h if isinstance(h,tuple) else (h,h)
        crop=F.normalize(self.crop_projection(hc).float(),dim=1)
        disease=F.normalize(self.disease_projection(hd).float(),dim=1)
        if getattr(self,'original_baseline',False):
            return None,self.backbone.crop.head(hc),self.backbone.disease.head(hd),crop,disease
        if self.family=='A':return None,self.crop_head(crop),self.disease_head(disease),crop,disease
        pair=F.normalize(self.pair_projection((hc+hd)/2).float(),dim=1)
        if not aligned:
            return None,self.crop_head((crop+pair)/2),self.disease_head((disease+pair)/2),crop,disease
        scale=self.log_temperature.clamp(math.log(1.),math.log(100.)).exp()
        return scale*pair@self.text0.T,scale*crop@self.text1.T,scale*disease@self.text2.T,crop,disease


def nonsemantic_banks(pairs,device,primitive=None):
    if primitive is None:
        generator=torch.Generator().manual_seed(shared.SEED)
        crop=F.normalize(torch.randn(shared.NUM_CROPS,512,generator=generator),dim=1).to(device)
        disease=F.normalize(torch.randn(shared.NUM_DISEASES,512,generator=generator),dim=1).to(device)
    else:crop,disease=primitive
    rows=pairs.to(device)
    pair=F.normalize(crop[rows[:,0]]+disease[rows[:,1]],dim=1)
    return [pair,crop,disease]
def construct(config,family,recipe,pairs,names,device,allow_download,backbone_factory=Dino,text_factory=TextOnly,meta=None):
    head_seed=shared.SEED
    backbone=backbone_factory(config,device,allow_download,revision=(meta or {}).get('dino',{}).get('revision'))
    blocks=0 if recipe.get('frozen') else config['partial_blocks'];backbone.policy(blocks,config['gradient_checkpointing'])
    text=None;banks=None;prompts=None;dimension=512
    if family=='B' and recipe.get('aligned',True):
        if recipe.get('semantic',True):
            text=text_factory(config,device,allow_download,revision=(meta or {}).get('text',{}).get('revision'))
            banks,prompts=text_banks(text,pairs,names,config,recipe.get('ensemble',False));dimension=text.dimension
        else:
            banks=nonsemantic_banks(pairs,device)
            prompts=dict(source='fixed nonsemantic random primitive codes, common.SEED; same branch routing/losses/scorer as semantic counterpart')
    # Loading a text model must not give B's matched visual projections a
    # different initialization simply by consuming the RNG first.
    torch.manual_seed(head_seed)
    model=Model(backbone,family,config,dimension,banks).to(device)
    if recipe.get('original_baseline'):
        model.requires_grad_(False);model.original_baseline=True
        model.backbone.crop.head.requires_grad_(True);model.backbone.disease.head.requires_grad_(True)
    elif family=='B' and recipe.get('aligned',True):
        model.crop_head.requires_grad_(False);model.disease_head.requires_grad_(False)
        if recipe['pair']==0:model.pair_projection.requires_grad_(False)
        if recipe['primitive']==0:
            model.crop_projection.requires_grad_(False)
            if not recipe.get('sc',0):model.disease_projection.requires_grad_(False)
    if hasattr(backbone,'augmentation_rng_state'):torch.set_rng_state(backbone.augmentation_rng_state)
    return model,text,prompts,dict(dino=backbone.metadata,text=text.metadata if text else None,blocks=blocks)


In [ ]:
FAMILY = 'A'
MODEL_NAME = 'approach3a_dino_disentangle_supcon'
print(MODEL_NAME, MODEL)


## 4. Training
Formal settings match ToTnki. Only method losses/heads/prototype source differ from the appropriate control. Common validation Top1 selects the checkpoint; earliest epoch breaks ties. No test-set tuning. Common batches may contain no cross-crop positives; that fraction is logged, never repaired with a different sampler. No formal result is guaranteed to improve.


In [ ]:
def positive_mask(crops, diseases, cross_crop=True):
    result = diseases[:, None] == diseases[None, :]
    result &= ~torch.eye(len(crops), dtype=torch.bool, device=crops.device)
    if cross_crop: result &= crops[:, None] != crops[None, :]
    return result


def supcon(features, crops, diseases, temperature=.1, cross_crop=True):
    features = F.normalize(features.float(), dim=1)
    positives = positive_mask(crops, diseases, cross_crop)
    valid = positives.any(1)
    if not valid.any(): return features.sum()*0, 0.
    diagonal = torch.eye(len(features), dtype=torch.bool, device=features.device)
    logits = (features @ features.T / temperature).masked_fill(diagonal, -torch.inf)
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    # masked_fill avoids 0 * -inf on the diagonal.
    means = log_prob.masked_fill(~positives, 0).sum(1) / positives.sum(1).clamp_min(1)
    return -means[valid].mean(), valid.float().mean().item()


def decorrelation(crop, disease):
    if len(crop) < 2: return (crop.sum()+disease.sum())*0
    crop, disease = crop.float(), disease.float()
    def standardize(x): return (x-x.mean(0)) / x.std(0, unbiased=True).clamp_min(1e-6)
    covariance = standardize(crop).T @ standardize(disease) / (len(crop)-1)
    return covariance.square().mean()


def pair_scores(crop, disease, pairs):
    pairs = pairs.to(crop.device)
    return crop.log_softmax(1)[:, pairs[:, 0]] + disease.log_softmax(1)[:, pairs[:, 1]]


def zscore(x):
    return (x-x.mean(1, keepdim=True)) / x.std(1, unbiased=False, keepdim=True).clamp_min(1e-6)


def factorized(parts, pairs, weights):
    pair, crop, disease = parts
    if len(weights) != 3 or min(weights) < 0 or abs(sum(weights)-1) > 1e-6:
        raise ValueError('Three nonnegative scorer weights summing to one required.')
    pairs = pairs.to(pair.device)
    return weights[0]*zscore(pair)+weights[1]*zscore(crop)[:, pairs[:,0]]+weights[2]*zscore(disease)[:, pairs[:,1]]
RECIPES = {
 'A': {
  'baseline': dict(sc=0.,dis=0.,original_baseline=True),
  'ce': dict(sc=0.,dis=0.),
  'disentangle': dict(sc=0.,dis=.01),
  'crosscrop': dict(sc=.1,dis=0.),
  'full': dict(sc=.1,dis=.01),
  'same_disease': dict(sc=.1,dis=.01,cross_crop=False)},
 'B': {
  'baseline': dict(pair=0.,primitive=0.,aligned=False,original_baseline=True),
  'visual_ce': dict(pair=1.,primitive=1.,semantic=False),
  'control_pair': dict(pair=1.,primitive=0.,semantic=False),
  'control_primitive': dict(pair=0.,primitive=1.,semantic=False),
  'control_crosscrop': dict(pair=1.,primitive=1.,sc=.1,semantic=False),
  'pair_only': dict(pair=1.,primitive=0.),
  'primitive_only': dict(pair=0.,primitive=1.),
  'full': dict(pair=1.,primitive=1.),
  'crosscrop': dict(pair=1.,primitive=1.,sc=.1),
  'ensemble': dict(pair=1.,primitive=1.,ensemble=True)}
}


def matched_control(family,variant):
    if variant=='baseline':return None
    if family=='A':return None if variant=='ce' else 'ce'
    return {'pair_only':'control_pair','primitive_only':'control_primitive','crosscrop':'control_crosscrop','full':'visual_ce','ensemble':'visual_ce'}.get(variant)


def metrics(scores,labels):
    scores=scores.float().cpu();labels=labels.long().cpu()
    if scores.ndim!=2 or len(scores)!=len(labels) or not len(labels) or not torch.isfinite(scores).all():
        raise ValueError('Invalid score matrix or empty split.')
    return dict(samples=len(labels),top1=top_k_accuracy(scores,labels,1),top5=top_k_accuracy(scores,labels,min(5,scores.shape[1])))


def validation_report(scores,frame):
    return metrics(scores,torch.tensor(frame.pair_row.to_numpy()))


def recipe_weights(recipe,config):
    if not recipe.get('aligned',True):return None
    if recipe.get('pair',0)==0:return [0.,.5,.5]
    if recipe.get('primitive',0)==0:return [1.,0.,0.]
    return config['score_grid'][0]


def score(output,pairs,family,recipe,weights=None):
    pair,crop,disease=output[:3]
    if family=='A' or not recipe.get('aligned',True):return pair_scores(crop,disease,pairs)
    return factorized((pair,crop,disease),pairs,weights)


def loss(output,labels,pairs,rows,family,recipe,config):
    _,crop,disease,uc,ud=output
    concepts=pairs.to(labels.device)[labels]
    zero=uc.sum()*0;ce_crop=ce_disease=pair_loss=zero
    if family=='A' or not recipe.get('aligned',True):
        ce_crop=F.cross_entropy(crop,concepts[:,0],label_smoothing=config['label_smoothing']);ce_disease=F.cross_entropy(disease,concepts[:,1],label_smoothing=config['label_smoothing'])
        value=ce_crop+ce_disease
    else:
        target_map={row:i for i,row in enumerate(rows)}
        try:targets=torch.tensor([target_map[int(i)] for i in labels],device=labels.device)
        except KeyError as error:raise ValueError('Proxy-held-out pair used as training target.') from error
        if recipe['pair']:pair_loss=F.cross_entropy(output[0][:,rows],targets,label_smoothing=config['label_smoothing'])
        if recipe['primitive']:
            ce_crop=F.cross_entropy(crop,concepts[:,0],label_smoothing=config['label_smoothing']);ce_disease=F.cross_entropy(disease,concepts[:,1],label_smoothing=config['label_smoothing'])
        value=recipe['pair']*pair_loss+recipe['primitive']*(ce_crop+ce_disease)
    sc,anchor=supcon(ud,concepts[:,0],concepts[:,1],config['supcon_temperature'],recipe.get('cross_crop',True))
    dis=decorrelation(uc,ud)
    value=value+recipe.get('sc',0.)*sc+recipe.get('dis',0.)*dis
    return value,dict(crop_ce=float(ce_crop.detach()),disease_ce=float(ce_disease.detach()),pair_ce=float(pair_loss.detach()),
        supcon=float(sc.detach()),decorrelation=float(dis.detach()),valid_anchor_fraction=anchor)


@torch.no_grad()
def collect(model,frame,config,pairs,family,recipe,weights,seed=None,perturb=None):
    model.eval();scores=[];parts=[[],[],[]];embeddings=[]
    for pixels,_,_,_ in loader(frame,config,seed=seed):
        if perturb is not None:pixels=perturb(pixels)
        output=model(pixels.to(next(model.parameters()).device),aligned=family=='B' and recipe.get('aligned',True))
        scores.append(score(output,pairs,family,recipe,weights).float().cpu())
        for store,value in zip(parts,output[:3]):
            if value is not None:store.append(value.float().cpu())
        embeddings.append(torch.cat(output[3:],dim=1).float().cpu())
    return torch.cat(scores),tuple(torch.cat(p) if p else None for p in parts),torch.cat(embeddings)


class Experiment:
    def __init__(self,family,config,mode='smoke',output=None,allow_download=False):
        if family not in RECIPES or mode not in ('smoke','full'):raise ValueError('Choose A/B and smoke/full.')
        self.family=family;self.config=copy.deepcopy(config);self.mode=mode;self.allow_download=allow_download
        self.initial_code_hashes=code_hashes(family)
        config=copy.deepcopy(config)
        if config['protocol']!='323-development-333-final' or config['image_backbone']!='vit_base_patch14_dinov2.lvd142m':raise ValueError('Wrong controlled protocol/backbone.')
        if any(k in config for k in ['e1_path','resume','checkpoint_path']):raise ValueError('Old model inputs are forbidden.')
        if not config['variants'] or set(config['variants'])-set(RECIPES[family]):raise ValueError('Unknown ablation.')
        control='ce' if family=='A' else 'visual_ce'
        if control not in config['variants']:raise ValueError('Include the matched CE control for every method comparison.')
        if family=='B':
            for variant in config['variants']:
                matched=matched_control(family,variant)
                if matched and matched not in config['variants']:raise ValueError('Missing matched control '+matched+' for '+variant)
        if config.get('seeds',[shared.SEED])!=[shared.SEED]:raise ValueError('Seeds must come from common.SEED, not a separate list.')
        config=copy.deepcopy(config);config['seeds']=[shared.SEED];self.config['seeds']=[shared.SEED]
        if len(set(config['variants']))!=len(config['variants']) or len(set(config['seeds']))!=len(config['seeds']):raise ValueError('Duplicate trials.')
        for k in ['epochs','batch_size','accumulation_steps','hidden_dim','feature_dim','text_batch_size']:
            if not isinstance(config[k],int) or config[k]<1:raise ValueError('Invalid '+k)
        if config['batch_size']<2 or config['partial_blocks']!=0:raise ValueError('Invalid batch/policy.')
        if config['precision'] not in ('float32','bf16'):raise ValueError('Choose float32/bf16 explicitly.')
        if not config['seeds'] or not all(isinstance(s,int) and s>=0 for s in config['seeds']):raise ValueError('Invalid seeds.')
        for k in ['crop_lr','disease_lr','weight_decay','supcon_temperature']:
            if not math.isfinite(config[k]) or config[k]<=0:raise ValueError('Invalid '+k)
        for weights in config['score_grid']:
            if len(weights)!=3 or any(not math.isfinite(w) or w<0 for w in weights) or abs(sum(weights)-1)>1e-6:raise ValueError('Invalid scoring grid.')
        if len(config['score_grid'])!=1:raise ValueError('Use one predeclared scorer; no method-specific scoring search.')
        if not 0<=config['label_smoothing']<1:raise ValueError('Invalid label smoothing.')
        if mode=='full':
            required=dict(epochs=20,batch_size=16,accumulation_steps=1,precision='float32',partial_blocks=0,
                          crop_lr=1e-4,disease_lr=3e-4,weight_decay=1e-4,label_smoothing=.1,gradient_clip=None)
            for key,value in required.items():
                if config.get(key)!=value:raise ValueError('Formal baseline setting must remain '+key+'='+str(value))
        if mode=='smoke':self.config.update(epochs=1,batch_size=8,accumulation_steps=1,precision='float32')
        self.device=shared.DEVICE
        if mode=='full' and self.device.type!='cuda':raise RuntimeError('Formal training requires school CUDA GPU.')
        if self.device.type=='cpu':torch.set_num_threads(4)
        if self.config['precision']=='bf16' and not torch.cuda.is_bf16_supported():raise RuntimeError('BF16 unsupported; select float32 before a new run.')
        self.frames,self.pairs,self.names,self.manifest=development()
        self.frozen_config=copy.deepcopy(self.config)
        base=Path(output or shared.RESULTS_DIR/'dino_controlled_common');base.mkdir(parents=True,exist_ok=True)
        # Both notebooks reuse an identical, immutable split manifest.
        self.split_path=base/'shared_split_manifest.json'
        if self.split_path.exists():
            if read(self.split_path)!=self.manifest:raise ValueError('Shared manifest changed; do not mix protocols.')
        else:write(self.split_path,self.manifest)
        self.path=base/(family+'_'+mode+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid4().hex[:8]);self.path.mkdir()
        write(self.path/'config.json',self.config);write(self.path/'environment.json',environment())
        write(self.path/'split_manifest.json',self.manifest)
        if mode=='full':
            hashes=duplicate_audit(self.frames)
            write(self.path/'training_image_hashes.json',hashes)
        self.completed=[];self.started=set()

    def run(self,variant,seed,backbone_factory=Dino,text_factory=TextOnly):
        if self.config!=self.frozen_config:raise ValueError('Settings changed; create a new session.')
        if code_hashes(self.family)!=self.initial_code_hashes:raise ValueError('Implementation changed during the session; start again.')
        if variant not in self.config['variants'] or seed not in self.config['seeds']:raise ValueError('Trial was not predeclared.')
        key=f'{variant}_seed{seed}'
        if key in self.started or (self.path/'frozen_eval_manifest.json').exists():raise RuntimeError('Already started/sealed; no overwrite or resume.')
        self.started.add(key);target=self.path/key;target.mkdir()
        seed_all(seed);recipe=copy.deepcopy(RECIPES[self.family][variant]);config=self.config
        fit=self.frames['fit'];validation=self.frames['validation']
        if self.mode=='smoke':fit=fit.iloc[:4];validation=validation.iloc[:4]
        model=text=None
        try:
            model,text,prompts,metadata=construct(config,self.family,recipe,self.pairs,self.names,self.device,self.allow_download,backbone_factory,text_factory)
            write(target/'model_metadata.json',metadata)
            if prompts:write(target/'development_prompts.json',prompts)
            names=[n for n,p in model.named_parameters() if p.requires_grad]
            frozen=[n for n,p in model.named_parameters() if not p.requires_grad]
            initial_heads=hashlib.sha256()
            for name,value in model.named_parameters():
                if not name.startswith('backbone.'):
                    initial_heads.update(name.encode());initial_heads.update(value.detach().cpu().contiguous().numpy().tobytes())
            write(target/'parameters.json',dict(trainable=names,frozen=frozen,initial_head_sha256=initial_heads.hexdigest(),
                trainable_count=sum(p.numel() for p in model.parameters() if p.requires_grad)))
            control=dict(backbone=metadata['dino'],split=fingerprint(self.manifest),seeds=config['seeds'],batch_size=config['batch_size'],
                epochs=config['epochs'],accumulation=config['accumulation_steps'],optimizer='AdamW',lr=config['crop_lr'],backbone_lr=0.,
                weight_decay=config['weight_decay'],precision=config['precision'],partial_blocks=config['partial_blocks'],
                gradient_clip=config['gradient_clip'],gradient_checkpointing=config['gradient_checkpointing'],hidden_dim=config['hidden_dim'],
                sampler='common RandomSampler, no custom batch construction',label_smoothing=config['label_smoothing'],scorer_search='none; predeclared scores for every method',augmentation='common get_transforms(imagenet); no override',crop_lr=config['crop_lr'],disease_lr=config['disease_lr'],common_seed=shared.SEED)
            write(target/'controlled_contract.json',dict(contract=control,sha256=fingerprint(control),frozen_ablation=bool(recipe.get('frozen'))))
            crop_params=[];disease_params=[]
            for name,p in model.named_parameters():
                if p.requires_grad:
                    (disease_params if ('disease' in name or 'disease_projection' in name) else crop_params).append(p)
            groups=[dict(params=crop_params,lr=config['crop_lr']),dict(params=disease_params,lr=config['disease_lr'])]
            optimizer=torch.optim.AdamW(groups,weight_decay=config['weight_decay'])
            scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=config['epochs'])
            rows=sorted(set(self.frames['fit'].pair_row));history=[];best=None;weights=recipe_weights(recipe,config) if self.family=='B' else None
            if self.device.type=='cuda':torch.cuda.reset_peak_memory_stats()
            train_batches=loader(fit,config,True)
            for epoch in range(config['epochs']):
                learning_rates=[group['lr'] for group in optimizer.param_groups]
                model.train()
                if recipe.get('frozen') or config['partial_blocks']==0:model.backbone.eval()
                optimizer.zero_grad(set_to_none=True);epoch_start=time.perf_counter();totals={};count=0
                batches=train_batches;steps=len(batches)
                for index,(pixels,_,_,labels) in enumerate(batches):
                    amp=torch.autocast(self.device.type,dtype=torch.bfloat16,enabled=config['precision']=='bf16')
                    with amp:
                        output=model(pixels.to(self.device),aligned=self.family=='B' and recipe.get('aligned',True))
                        value,details=loss(output,labels.to(self.device),self.pairs,rows,self.family,recipe,config)
                    if not torch.isfinite(value):raise RuntimeError('Nonfinite loss.')
                    divisor=min(config['accumulation_steps'],steps-(index//config['accumulation_steps'])*config['accumulation_steps'])
                    (value/divisor).backward()
                    if text:text.assert_frozen()
                    if (index+1)%config['accumulation_steps']==0 or index+1==steps:
                        gradients=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
                        if not gradients or not all(torch.isfinite(g).all() for g in gradients):raise RuntimeError('Invalid gradients.')
                        if any(p.grad is not None for p in model.backbone.parameters() if not p.requires_grad):raise RuntimeError('Frozen DINO gradients.')
                        if config['gradient_clip'] is not None:torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],config['gradient_clip'])
                        optimizer.step();optimizer.zero_grad(set_to_none=True)
                    details['total_loss']=float(value.detach())
                    for k,v in details.items():totals[k]=totals.get(k,0.)+v*len(labels)
                    count+=len(labels)
                scheduler.step()
                a,_,_=collect(model,validation,config,self.pairs,self.family,recipe,weights,seed)
                report=validation_report(a,validation)
                record=dict(epoch=epoch+1,learning_rates_used=learning_rates,losses={k:v/count for k,v in totals.items()},validation=report,seconds=time.perf_counter()-epoch_start,
                    peak_gpu_bytes=torch.cuda.max_memory_allocated() if self.device.type=='cuda' else None)
                history.append(record);print(key,record,flush=True)
                selection=report['top1']
                if best is None or selection>best:
                    best=selection;checkpoint=target/f'epoch_{epoch+1:03d}.pt'
                    with checkpoint.open('xb') as stream:torch.save(dict(state_dict={k:v.detach().cpu() for k,v in model.state_dict().items()},
                        epoch=epoch+1,family=self.family,variant=variant,recipe=recipe,seed=seed,config=config,pairs=self.pairs,
                        manifest_sha256=fingerprint(self.manifest),control=control,history=list(history),smoke=self.mode=='smoke'),stream)
                    winner=checkpoint
            saved=torch.load(winner,map_location='cpu',weights_only=True);model.load_state_dict(saved['state_dict'],strict=True)
            write(target/'history.json',history)
            # No method-specific scorer search: weights are predeclared, just as
            # the visual CE controls use a fixed crop+disease scoring rule.
            chosen=weights;calibration=[]
            write(target/'selection.json',dict(checkpoint=winner.name,checkpoint_sha256=sha(winner),recipe=recipe,weights=chosen,calibration=calibration,
                best_epoch=saved['epoch'],criterion='common validation Top1 over 323 candidates; earliest epoch on ties',
                optimization_images=len(fit),intended_optimization_images=len(self.frames['fit']),trained_on_common_7223=self.mode=='full'))
            self.completed.append(key);return target
        except Exception as error:
            write(target/'error.json',dict(type=type(error).__name__,message=str(error)));raise
        finally:
            del model,text;gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()

    def run_all(self,**factories):
        for seed in self.config['seeds']:
            for variant in self.config['variants']:self.run(variant,seed,**factories)
        return self.seal()

    def seal(self):
        expected={f'{v}_seed{s}' for s in self.config['seeds'] for v in self.config['variants']}
        if self.config!=self.frozen_config or set(self.completed)!=expected:raise RuntimeError('All predeclared trials must finish unchanged before final evaluation.')
        if code_hashes(self.family)!=self.initial_code_hashes:raise ValueError('Implementation changed during the session; cannot seal.')
        control='ce' if self.family=='A' else 'visual_ce'
        for seed in self.config['seeds']:
            initial=read(self.path/f'{control}_seed{seed}'/'parameters.json')['initial_head_sha256']
            for variant in self.config['variants']:
                if variant=='baseline':continue
                if read(self.path/f'{variant}_seed{seed}'/'parameters.json')['initial_head_sha256']!=initial:
                    raise ValueError('Method and matched control did not start from the same head initialization.')
        for seed in self.config['seeds']:
            for variant in self.config['variants']:
                matched=matched_control(self.family,variant)
                if matched:
                    a=read(self.path/f'{variant}_seed{seed}'/'parameters.json')['trainable']
                    b=read(self.path/f'{matched}_seed{seed}'/'parameters.json')['trainable']
                    if a!=b:raise ValueError('Matched control trainable paths differ: '+variant)
        paths=['config.json','split_manifest.json']
        if self.mode=='full':paths.append('training_image_hashes.json')
        for key in self.completed:
            selection=read(self.path/key/'selection.json')
            paths.extend([key+'/'+name for name in ['selection.json','model_metadata.json','controlled_contract.json','parameters.json',selection['checkpoint']]])
        write(self.path/'frozen_eval_manifest.json',dict(family=self.family,mode=self.mode,trials=self.completed,
            files={p:sha(self.path/p) for p in paths},recipes_sha256=fingerprint(RECIPES),
            train_split_sha256=fingerprint(self.manifest),code_sha256=self.initial_code_hashes,official_metadata_read=False))
        return self.path


In [ ]:
config = copy.deepcopy(CONFIG)
config['variants'] = ['baseline', 'ce', 'disentangle', 'crosscrop', 'full', 'same_disease']
if MODE == 'smoke':
    config['variants'] = ['baseline', 'ce', 'full']
experiment = Experiment(FAMILY, config, MODE, allow_download=ALLOW_DOWNLOAD)
print('Run folder:', experiment.path)
if RUN_TRAIN:
    RUN_PATH = experiment.run_all()
    print('Frozen run:', RUN_PATH)
else:
    RUN_PATH = None
    print('Audit only: no weights loaded and no training started.')


## 5. Evaluation
Finish and freeze BOTH formal notebooks before official testing. Set EXISTING_RUN and OTHER_RUN (or COMPARISON_LOCK) when reopening; flags are not part of algorithm hashes. Final verifies code/config/checkpoints and equal baseline contracts BEFORE opening test CSVs, then expands323->333. Smoke-final uses disjoint validation subsets and is NOT true unseen evaluation. Use common Top1/5/HM and robustness; never tune from this table.


In [ ]:
def verify(root):
    root=Path(root);seal=read(root/'frozen_eval_manifest.json')
    for relative,digest in seal['files'].items():
        if sha(root/relative)!=digest:raise ValueError('Frozen artifact changed: '+relative)
    if seal['code_sha256']!=code_hashes(seal['family']) or seal['recipes_sha256']!=fingerprint(RECIPES):raise ValueError('Frozen implementation changed.')
    frames,pairs,names,manifest=development()
    if fingerprint(manifest)!=seal['train_split_sha256']:raise ValueError('Development data changed.')
    return seal,read(root/'config.json'),frames,pairs,names


def choose_predeclared(root,seal):
    """Freeze one winner per seed before any test metadata is read; report all trials."""
    winners={}
    for key in seal['trials']:
        selection=read(root/key/'selection.json');saved=torch.load(root/key/selection['checkpoint'],map_location='cpu',weights_only=True)
        validation=saved['history'][-1]['validation'];rank=validation['top1']
        seed=str(saved['seed'])
        if seed not in winners or rank>winners[seed]['rank']:winners[seed]=dict(trial=key,rank=rank)
    return winners


def lock_comparison(first,second):
    roots=[Path(first).resolve(),Path(second).resolve()]
    if roots[0].parent!=roots[1].parent:raise ValueError('Keep both run directories under the same parent for a portable comparison lock.')
    seals=[verify(root)[0] for root in roots]
    if {s['family'] for s in seals}!={'A','B'} or len({s['mode'] for s in seals})!=1:raise ValueError('Lock one A and one B run of the same mode.')
    controls=[]
    for root,seal in zip(roots,seals):
        for key in seal['trials']:controls.append(read(root/key/'controlled_contract.json')['contract'])
    if any(c!=controls[0] for c in controls):raise ValueError('Backbone weights, preprocessing, split, seeds, or training budget differ.')
    path=roots[0].parent/('comparison_lock_'+uuid4().hex[:8]+'.json')
    write(path,dict(runs={seal['family']:dict(directory=root.name,seal_sha256=sha(root/'frozen_eval_manifest.json')) for root,seal in zip(roots,seals)},
        controlled_contract=controls[0],frozen_ablation_exception='Explicit frozen variants train zero visual blocks; do not mix with primary partial-policy claims'))
    return path


def perturbation(name):
    from common.evaluate import PERTURBATIONS
    if name not in PERTURBATIONS:raise ValueError('Use common robustness perturbations only.')
    mean=torch.tensor(shared.NORM_STATS['imagenet'][0]).view(1,3,1,1)
    std=torch.tensor(shared.NORM_STATS['imagenet'][1]).view(1,3,1,1)
    state=torch.Generator().manual_seed(shared.SEED).get_state()
    def apply(images):
        nonlocal state
        with torch.random.fork_rng(devices=[]):
            torch.set_rng_state(state)
            result=(PERTURBATIONS[name](images*std+mean)-mean)/std
            state=torch.get_rng_state()
        return result
    return apply


def report(scores,frames,pairs,fit):
    result={};details=[]
    for split in ('seen','unseen'):
        frame=frames[split];labels=torch.tensor(frame.pair_row.to_numpy());values=scores[split].float().cpu()
        result[split]=metrics(values,labels)
        predicted=pairs[values.argmax(1)];truth=pairs[labels]
        result[split].update(crop_top1=round(100*(predicted[:,0]==truth[:,0]).float().mean().item(),2),
                             disease_top1=round(100*(predicted[:,1]==truth[:,1]).float().mean().item(),2))
        for label in labels.unique(sorted=True):
            mask=labels==label;subset=metrics(values[mask],labels[mask])
            details.append(dict(split=split,pair_row=int(label),**subset))
    result.update(hm1=harmonic_mean(result['seen']['top1'],result['unseen']['top1']),
                  hm5=harmonic_mean(result['seen']['top5'],result['unseen']['top5']),candidate_count=len(pairs),per_class=details)
    all_frames=pd.concat([frames['seen'],frames['unseen']],ignore_index=True);values=torch.cat([scores['seen'],scores['unseen']])
    crop_seen=all_frames.crop.isin(fit.crop);disease_seen=all_frames.disease.isin(fit.disease)
    observed=set(zip(fit.crop,fit.disease));pair_seen=pd.Series([(c,d) in observed for c,d in zip(all_frames.crop,all_frames.disease)])
    result['ood']={}
    for name,mask in [('unseen_crop',~crop_seen),('unseen_disease',~disease_seen),('unseen_combination',crop_seen&disease_seen&~pair_seen)]:
        mask=torch.tensor(mask.to_numpy());labels=torch.tensor(all_frames.pair_row.to_numpy())[mask]
        result['ood'][name]=metrics(values[mask],labels) if mask.any() else dict(samples=0,top1=None,top5=None)
    return result


def prediction_rows(scores,frame,pairs,split):
    top1=scores.topk(1,dim=1).indices[:,0];top5=scores.topk(5,dim=1)
    rows=[]
    for i,row in enumerate(frame.itertuples(index=False)):
        chosen=int(top1[i]);c,d=pairs[chosen].tolist()
        rows.append(dict(image=row.image,split=split,true_pair_row=int(row.pair_row),true_crop=int(row.crop),true_disease=int(row.disease),
            predicted_pair_row=chosen,predicted_crop=c,predicted_disease=d,
            top5_pairs=json.dumps(top5.indices[i].tolist()),top5_scores=json.dumps(top5.values[i].tolist())))
    return rows


def confusion(path,scores,frame,pairs):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    matrix=torch.zeros(323,len(pairs),dtype=torch.long);labels=torch.tensor(frame.pair_row.to_numpy())
    matrix.index_put_((labels,scores.argmax(1)),torch.ones_like(labels),accumulate=True)
    pd.DataFrame(matrix.numpy()).to_csv(path/'confusion_seen.csv',index_label='true_pair_row')
    normalized=matrix.float()/matrix.sum(1,keepdim=True).clamp_min(1)
    fig,ax=plt.subplots(figsize=(12,10));plot=ax.imshow(normalized,aspect='auto',vmin=0,vmax=1)
    ax.set(xlabel='Predicted candidate row',ylabel='True seen pair row',title='Seen confusion: row-normalized; exact counts in CSV')
    fig.colorbar(plot,ax=ax);fig.savefig(path/'confusion_seen.png',dpi=130,bbox_inches='tight');plt.close(fig)


def final_evaluate(run_path,comparison_lock=None,allow_download=False,backbone_factory=Dino,text_factory=TextOnly):
    root=Path(run_path).resolve();seal,config,development_frames,dev_pairs,names=verify(root)
    if seal['mode']=='full':
        if comparison_lock is None:raise ValueError('Lock both A and B before any official test access.')
        lock_path=Path(comparison_lock).resolve();locked=read(lock_path)['runs'].get(seal['family'])
        if not locked or (lock_path.parent/locked['directory']).resolve()!=root or locked['seal_sha256']!=sha(root/'frozen_eval_manifest.json'):
            raise ValueError('Comparison lock does not match this frozen run.')
        # Verify BOTH runs remain frozen before this model can touch tests.
        lock_data=read(lock_path)
        if set(lock_data['runs'])!={'A','B'}:raise ValueError('Both A and B must be in the joint lock.')
        for family,item in lock_data['runs'].items():
            other=lock_path.parent/item['directory'];other_seal=verify(other)[0]
            if other_seal['family']!=family or other_seal['mode']!=seal['mode']:raise ValueError('Comparison method/stage changed.')
            if sha(other/'frozen_eval_manifest.json')!=item['seal_sha256']:raise ValueError('Other comparison run changed.')
            for trial in other_seal['trials']:
                if read(other/trial/'controlled_contract.json')['contract']!=lock_data['controlled_contract']:
                    raise ValueError('Backbone or controlled budget differs from the joint lock.')
    winners=choose_predeclared(root,seal)
    path=root/'evaluations'/('final_'+uuid4().hex[:8]);path.mkdir(parents=True)
    write(path/'evaluation_started.json',dict(winners_selected_on_validation=winners,run_seal_sha256=sha(root/'frozen_eval_manifest.json'),
        comparison_lock_sha256=sha(comparison_lock) if comparison_lock else None))
    if seal['mode']=='full':
        pairs,frames=official(dev_pairs)
        hashes=duplicate_audit(dict(training=pd.concat([development_frames['fit'],development_frames['validation']]),**frames))
        original=read(root/'training_image_hashes.json')['image_sha256']
        if any(hashes['image_sha256'][name]!=digest for name,digest in original.items()):raise ValueError('Training images changed since development.')
        write(path/'final_image_audit.json',dict(exact_hash_disjoint=True,checked_images=len(hashes['image_sha256'])))
    else:pairs=dev_pairs;frames=dict(seen=development_frames['validation'].iloc[:4],unseen=development_frames['validation'].iloc[4:8])
    write(path/'candidate_space.json',dict(pairs=pairs.tolist(),official_test_access=seal['mode']=='full'))
    summaries=[];inference_artifacts={}
    for key in seal['trials']:
        target=path/key;target.mkdir();selection=read(root/key/'selection.json');meta=read(root/key/'model_metadata.json')
        saved=torch.load(root/key/selection['checkpoint'],map_location='cpu',weights_only=True)
        if saved['smoke']!=(seal['mode']=='smoke') or not torch.equal(saved['pairs'],dev_pairs):raise ValueError('Wrong checkpoint stage/candidates.')
        seed_all(saved['seed']);device=torch.device('cuda' if torch.cuda.is_available() else 'cpu');model=text=None
        try:
            model,text,_,metadata=construct(config,seal['family'],selection['recipe'],dev_pairs,names,device,allow_download,backbone_factory,text_factory,meta)
            if metadata!=meta:raise ValueError('Pretrained model identity changed.')
            model.load_state_dict(saved['state_dict'],strict=True);del saved
            final_banks=None
            if text:
                final_banks,prompts=text_banks(text,pairs,names,config,selection['recipe'].get('ensemble',False))
                before={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
                model.install_text(final_banks)
                if any(not torch.equal(p,dict(model.named_parameters())[n]) for n,p in before.items()):raise RuntimeError('Text-query expansion changed model parameters.')
                write(target/'final_prompts.json',prompts)
            if seal['family']=='B' and selection['recipe'].get('aligned',True) and not selection['recipe'].get('semantic',True):
                final_banks=nonsemantic_banks(pairs,device,(model.text1,model.text2))
                model.install_text(final_banks)
            model.eval().requires_grad_(False)
            scores={split:collect(model,frame,config,pairs,seal['family'],selection['recipe'],selection['weights'])[0] for split,frame in frames.items()}
            result=report(scores,frames,pairs,development_frames['fit'])
            result.update(trial=key,smoke=seal['mode']=='smoke',optimization_images=selection['optimization_images'],
                intended_optimization_images=selection['intended_optimization_images'],trained_on_common_7223=seal['mode']=='full',
                candidate_order_sha256=fingerprint(pairs.tolist()),prediction_rule=selection,
                limitation='exploratory after historical test exposure; single-seed runs do not estimate training variance')
            robustness={}
            if seal['mode']=='full':
                for name in config['corruptions']:
                    changed=collect(model,frames['seen'],config,pairs,seal['family'],selection['recipe'],selection['weights'],perturb=perturbation(name))[0]
                    value=metrics(changed,torch.tensor(frames['seen'].pair_row.to_numpy()))
                    robustness[name]=dict(**value,drop=round(result['seen']['top1']-value['top1'],2))
            result['robustness']=robustness or dict(status='not_run_in_smoke')
            write(target/'metrics.json',result)
            pd.DataFrame([r for split in frames for r in prediction_rows(scores[split],frames[split],pairs,split)]).to_csv(target/'predictions.csv',index=False)
            pd.DataFrame(result['per_class']).to_csv(target/'per_class.csv',index=False)
            confusion(target,scores['seen'],frames['seen'],pairs)
            with (target/'scores.pt').open('xb') as stream:torch.save(dict(scores=scores,pairs=pairs,labels={k:torch.tensor(v.pair_row.to_numpy()) for k,v in frames.items()}),stream)
            with (target/'inference_queries.pt').open('xb') as stream:torch.save(dict(pairs=pairs,banks=[b.cpu() for b in final_banks] if final_banks else None),stream)
            inference_artifacts[key]=dict(query_sha256=sha(target/'inference_queries.pt'))
            summaries.append(dict(trial=key,variant=key.rsplit('_seed',1)[0],seed=int(key.rsplit('_seed',1)[1]),seen_top1=result['seen']['top1'],
                seen_top5=result['seen']['top5'],unseen_top1=result['unseen']['top1'],unseen_top5=result['unseen']['top5'],hm1=result['hm1'],hm5=result['hm5']))
        except Exception as error:
            write(target/'error.json',dict(type=type(error).__name__,message=str(error)));raise
        finally:
            del model,text;gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
    table=pd.DataFrame(summaries);table.to_csv(path/'comparison.csv',index=False)
    # Pair by seed against the predeclared matched control, never the best test score.
    deltas=[]
    for item in summaries:
        baseline=matched_control(seal['family'],item['variant'])
        if baseline is None:continue
        controls=[r for r in summaries if r['variant']==baseline and r['seed']==item['seed']]
        if len(controls)!=1:raise ValueError('Missing paired control result.')
        control=controls[0]
        row=dict(variant=item['variant'],seed=item['seed'],control=baseline,comparison_type='matched-method')
        for metric in ['seen_top1','seen_top5','unseen_top1','unseen_top5','hm1','hm5']:
            row[metric+'_delta_pp']=round(item[metric]-control[metric],4)
        deltas.append(row)
    pd.DataFrame(deltas).to_csv(path/'paired_method_deltas.csv',index=False)
    delta_summary=[]
    if deltas:
        for variant,group in pd.DataFrame(deltas).groupby('variant',sort=False):
            summary=dict(variant=variant,seeds=len(group),interpretation='paired differences; not proof of significance or guaranteed improvement')
            for metric in ['seen_top1','unseen_top1','hm1']:
                values=group[metric+'_delta_pp'];summary[metric+'_mean_delta_pp']=float(values.mean())
                summary[metric+'_std_delta_pp']=float(values.std(ddof=1)) if len(values)>1 else None
                summary[metric+'_positive_seeds']=int((values>0).sum())
            delta_summary.append(summary)
    write(path/'paired_method_summary.json',delta_summary)
    aggregates=[]
    for name,group in table.groupby('variant',sort=False):
        row=dict(variant=name,seeds=len(group))
        for metric in ['seen_top1','seen_top5','unseen_top1','unseen_top5','hm1','hm5']:
            row[metric+'_mean']=float(group[metric].mean());row[metric+'_std']=float(group[metric].std(ddof=1)) if len(group)>1 else None
        aggregates.append(row)
    write(path/'seed_summary.json',aggregates)
    write(path/'completed.json',dict(run_seal_sha256=sha(root/'frozen_eval_manifest.json'),inference=inference_artifacts,smoke=seal['mode']=='smoke'))
    return path


In [ ]:
EVALUATION_PATH = None
if RUN_FINAL:
    selected_run = Path(EXISTING_RUN) if EXISTING_RUN else RUN_PATH
    if selected_run is None:
        raise ValueError('Finish development or set EXISTING_RUN first.')
    lock = COMPARISON_LOCK
    if MODE == 'full' and lock is None and OTHER_RUN is not None:
        lock = lock_comparison(selected_run, Path(OTHER_RUN))
        print('Comparison lock:', lock)
    EVALUATION_PATH = final_evaluate(selected_run, lock, allow_download=ALLOW_DOWNLOAD)
    print('Evaluation folder:', EVALUATION_PATH)


## 6. Save
Training already saved unique recovery checkpoints; evaluation saved metrics, predictions, scores and confusion files. Keep the entire run parent folder on persistent school storage. comparison.csv reports all variants. paired_method_deltas.csv is method minus its matched control; negative deltas remain. A single common seed is not statistical significance. Empty unseen-crop/disease groups are N/A. Optional inference helpers below index only common training images.


In [ ]:
def load_predictor(run_path,evaluation_path,trial,allow_download=False,backbone_factory=Dino):
    root=Path(run_path).resolve();evaluation=Path(evaluation_path).resolve();seal,config,_,pairs,_=verify(root)
    done=read(evaluation/'completed.json')
    if done['run_seal_sha256']!=sha(root/'frozen_eval_manifest.json') or trial not in seal['trials']:raise ValueError('Inference contract mismatch.')
    queries=evaluation/trial/'inference_queries.pt'
    if sha(queries)!=done['inference'][trial]['query_sha256']:raise ValueError('Inference queries changed.')
    data=torch.load(queries,map_location='cpu',weights_only=True);selection=read(root/trial/'selection.json')
    meta=read(root/trial/'model_metadata.json');device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    backbone=backbone_factory(config,device,allow_download,revision=meta['dino']['revision'])
    if backbone.metadata!=meta['dino']:raise ValueError('Backbone identity changed.')
    backbone.policy(0,False)
    saved=torch.load(root/trial/selection['checkpoint'],map_location='cpu',weights_only=True)
    old_banks=[saved['state_dict']['text'+str(i)] for i in range(3)] if data['banks'] else None
    dimension=old_banks[0].shape[1] if old_banks else 512
    model=Model(backbone,seal['family'],config,dimension,old_banks).to(device)
    model.load_state_dict(saved['state_dict'],strict=True)
    model.original_baseline=bool(selection['recipe'].get('original_baseline'))
    if data['banks']:model.install_text([b.to(device) for b in data['banks']])
    model.eval().requires_grad_(False)
    class Predictor:
        candidate_pairs=data['pairs']
        @torch.no_grad()
        def __call__(self,images):
            result=model(images.to(device),aligned=seal['family']=='B' and selection['recipe'].get('aligned',True))
            return score(result,self.candidate_pairs,seal['family'],selection['recipe'],selection['weights']).float().cpu()
        @torch.no_grad()
        def embed(self,images):
            output=model(images.to(device),aligned=seal['family']=='B' and selection['recipe'].get('aligned',True))
            return F.normalize(torch.cat(output[3:],dim=1).float(),dim=1).cpu()
    return Predictor()


def training_retrieval_index(run_path,predictor):
    seal,config,frames,_,_=verify(run_path);features=[]
    for images,*_ in loader(frames['fit'],config):features.append(predictor.embed(images))
    return dict(features=torch.cat(features),images=frames['fit'].image.tolist(),source='common optimization-training images only')


def nearest_training(query_embeddings,index,k=5):
    similarities=F.normalize(query_embeddings.float(),dim=1)@F.normalize(index['features'].float(),dim=1).T
    values,ids=similarities.topk(min(k,len(index['images'])),dim=1)
    return [[dict(image=index['images'][int(i)],cosine=float(v)) for i,v in zip(row,scores)] for row,scores in zip(ids,values)]


In [ ]:
print('Keep this complete run folder:', experiment.path)
if EVALUATION_PATH is not None:
    display(pd.read_csv(EVALUATION_PATH / 'comparison.csv'))
    display(pd.read_csv(EVALUATION_PATH / 'paired_method_deltas.csv'))
    print('Save the shared split manifest, joint lock and all trial checkpoints too.')
